In [0]:
%run ../../02_common_utils/operations

In [0]:
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS staging")
spark.sql("USE SCHEMA staging")

from pyspark.sql.functions import (
    col, trim, lit, when, to_date,
    md5, concat_ws, current_timestamp
)
from pyspark.sql.types import (
    LongType, IntegerType, DecimalType
)
from datetime import datetime

team_name   = "team_lemma"
bronze_db   = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db   = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"

try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {bronze_db}.finwire LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "Batch1"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "Batch1"

print(f"bronze  : {bronze_db}")
print(f"staging : {staging_db}")
print(f"run_id  : {carried_run_id}")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'bronze_to_staging_market_finwire', f'Starting processing for FINWIRE parsing (batch: {carried_batch})')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'RUNNING')

In [0]:
from pyspark.sql.functions import expr

def safe_date(column_name: str, fmt: str = "yyyyMMdd"):
    """
    Null-safe date cast — returns NULL for empty / unparseable strings
    instead of raising CANNOT_PARSE_TIMESTAMP.
    Uses Spark SQL try_to_date() under the hood.
    """
    return expr(f"try_to_date(trim({column_name}), '{fmt}')")


def safe_eff_date():
    return expr("try_to_date(trim(substring(PTS, 1, 8)), 'yyyyMMdd')").alias("EffectiveDate")

In [0]:
bronze_fw = spark.table(f"{bronze_db}.finwire")

NULL_STR  = lit(None).cast("string")
NULL_LONG = lit(None).cast(LongType())
NULL_INT  = lit(None).cast(IntegerType())
NULL_DT   = lit(None).cast("date")
NULL_D102 = lit(None).cast(DecimalType(10, 2))
NULL_D152 = lit(None).cast(DecimalType(15, 2))

is_cik = trim(col("CoNameOrCIK")).rlike(r"^\d+$")

cmp_df = (
    bronze_fw
    .filter(trim(col("RecType")) == "CMP")
    .select(
        lit("CMP").alias("RecType"),
        safe_eff_date(),                                         
        trim(col("PTS")).alias("PTS"),
        trim(col("CompanyName")).alias("CompanyName"),
        trim(col("CIK")).cast(LongType()).alias("CompanyID"),
        trim(col("Status")).alias("Status"),
        trim(col("IndustryID")).alias("IndustryID"),
        trim(col("SPrating")).alias("SPrating"),
        safe_date("FoundingDate").alias("FoundingDate"),         
        trim(col("AddrLine1")).alias("AddrLine1"),
        trim(col("AddrLine2")).alias("AddrLine2"),
        trim(col("PostalCode")).alias("PostalCode"),
        trim(col("City")).alias("City"),
        trim(col("StateProvince")).alias("StateProvince"),
        trim(col("Country")).alias("Country"),
        trim(col("CEOname")).alias("CEOname"),
        trim(col("Description")).alias("Description"),
        NULL_STR.alias("Symbol"),       NULL_STR.alias("IssueType"),
        NULL_STR.alias("Name"),         NULL_STR.alias("ExID"),
        NULL_LONG.alias("ShOut"),       NULL_DT.alias("FirstTradeDate"),
        NULL_DT.alias("FirstTradeExchg"), NULL_D102.alias("Dividend"),
        NULL_STR.alias("CoNameOrCIK"),
        NULL_INT.alias("FIYear"),       NULL_INT.alias("FIQtr"),
        NULL_DT.alias("QtrStartDate"),  NULL_DT.alias("PostingDate"),
        NULL_D152.alias("Revenue"),     NULL_D152.alias("Earnings"),
        NULL_D102.alias("EPS"),         NULL_D102.alias("DilutedEPS"),
        NULL_D102.alias("Margin"),      NULL_D152.alias("Inventory"),
        NULL_D152.alias("Assets"),      NULL_D152.alias("Liabilities"),
        NULL_LONG.alias("DilutedShOut"),
        col("_batch"), lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
    )
)
print(f"CMP rows : {cmp_df.count():,}")

In [0]:
#  SEC 
sec_df = (
    bronze_fw
    .filter(trim(col("RecType")) == "SEC")
    .withColumn("RecType5",
        when(is_cik, "SEC_CIK").otherwise("SEC_NAME"))
    .select(
        col("RecType5").alias("RecType"),
        safe_eff_date(),                                          
        trim(col("PTS")).alias("PTS"),
        NULL_STR.alias("CompanyName"),  NULL_LONG.alias("CompanyID"),
        trim(col("Status")).alias("Status"),
        NULL_STR.alias("IndustryID"),   NULL_STR.alias("SPrating"),
        NULL_DT.alias("FoundingDate"),
        NULL_STR.alias("AddrLine1"),    NULL_STR.alias("AddrLine2"),
        NULL_STR.alias("PostalCode"),   NULL_STR.alias("City"),
        NULL_STR.alias("StateProvince"),NULL_STR.alias("Country"),
        NULL_STR.alias("CEOname"),      NULL_STR.alias("Description"),
        trim(col("Symbol")).alias("Symbol"),
        trim(col("IssueType")).alias("IssueType"),
        trim(col("Name")).alias("Name"),
        trim(col("ExID")).alias("ExID"),
        trim(col("ShOut")).cast(LongType()).alias("ShOut"),
        safe_date("FirstTradeDate").alias("FirstTradeDate"),      # ← fixed
        safe_date("FirstTradeExchg").alias("FirstTradeExchg"),    # ← fixed
        trim(col("Dividend")).cast(DecimalType(10, 2)).alias("Dividend"),
        trim(col("CoNameOrCIK")).alias("CoNameOrCIK"),
        NULL_INT.alias("FIYear"),       NULL_INT.alias("FIQtr"),
        NULL_DT.alias("QtrStartDate"),  NULL_DT.alias("PostingDate"),
        NULL_D152.alias("Revenue"),     NULL_D152.alias("Earnings"),
        NULL_D102.alias("EPS"),         NULL_D102.alias("DilutedEPS"),
        NULL_D102.alias("Margin"),      NULL_D152.alias("Inventory"),
        NULL_D152.alias("Assets"),      NULL_D152.alias("Liabilities"),
        NULL_LONG.alias("DilutedShOut"),
        col("_batch"), lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
    )
)

#  FIN 
fin_df = (
    bronze_fw
    .filter(trim(col("RecType")) == "FIN")
    .withColumn("RecType5",
        when(is_cik, "FIN_COMPANYID").otherwise("FIN_NAME"))
    .select(
        col("RecType5").alias("RecType"),
        safe_eff_date(),                                          
        trim(col("PTS")).alias("PTS"),
        NULL_STR.alias("CompanyName"),  NULL_LONG.alias("CompanyID"),
        NULL_STR.alias("Status"),       NULL_STR.alias("IndustryID"),
        NULL_STR.alias("SPrating"),     NULL_DT.alias("FoundingDate"),
        NULL_STR.alias("AddrLine1"),    NULL_STR.alias("AddrLine2"),
        NULL_STR.alias("PostalCode"),   NULL_STR.alias("City"),
        NULL_STR.alias("StateProvince"),NULL_STR.alias("Country"),
        NULL_STR.alias("CEOname"),      NULL_STR.alias("Description"),
        NULL_STR.alias("Symbol"),       NULL_STR.alias("IssueType"),
        NULL_STR.alias("Name"),         NULL_STR.alias("ExID"),
        trim(col("ShOut")).cast(LongType()).alias("ShOut"),
        NULL_DT.alias("FirstTradeDate"),NULL_DT.alias("FirstTradeExchg"),
        NULL_D102.alias("Dividend"),
        trim(col("CoNameOrCIK")).alias("CoNameOrCIK"),
        trim(col("Year")).cast(IntegerType()).alias("FIYear"),
        trim(col("Quarter")).cast(IntegerType()).alias("FIQtr"),
        safe_date("QtrStartDate").alias("QtrStartDate"),          # ← fixed
        safe_date("PostingDate").alias("PostingDate"),            # ← fixed
        trim(col("Revenue")).cast(DecimalType(15, 2)).alias("Revenue"),
        trim(col("Earnings")).cast(DecimalType(15, 2)).alias("Earnings"),
        trim(col("EPS")).cast(DecimalType(10, 2)).alias("EPS"),
        trim(col("DilutedEPS")).cast(DecimalType(10, 2)).alias("DilutedEPS"),
        trim(col("Margin")).cast(DecimalType(10, 2)).alias("Margin"),
        trim(col("Inventory")).cast(DecimalType(15, 2)).alias("Inventory"),
        trim(col("Assets")).cast(DecimalType(15, 2)).alias("Assets"),
        trim(col("Liabilities")).cast(DecimalType(15, 2)).alias("Liabilities"),
        trim(col("DilutedShOut")).cast(LongType()).alias("DilutedShOut"),
        col("_batch"), lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
    )
)

print(f"SEC rows : {sec_df.count():,}")
print(f"FIN rows : {fin_df.count():,}")


In [0]:

finwire_parsed = cmp_df.unionByName(sec_df).unionByName(fin_df)

(finwire_parsed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("RecType")
    .saveAsTable(f"{staging_db}.finwire_parsed"))

print(" staging.finwire_parsed written")

In [0]:
result = (
    spark.table(f"{staging_db}.finwire_parsed")
    .groupBy("RecType")
    .count()
    .orderBy("RecType")
)
display(result)



total = spark.table(f"{staging_db}.finwire_parsed").count()
print(f"\nTotal: {total:,}  (expected 470,025)")

In [0]:
%run ../../02_common_utils/operations

In [0]:
##  staging finwire audit log 
source_count = spark.table(f"{bronze_db}.finwire").count()
target_count = spark.table(f"{staging_db}.finwire_parsed").count()
carried_run_id = str(
    spark.table(f"{staging_db}.finwire_parsed").select("`_run_id`").first()[0]
)

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="Batch1",
    domain="MARKET",
    table_name="finwire_parsed",
    source_layer="bronze",
    target_layer="staging",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="Batch1",
    layer="staging",
    table_name="finwire_parsed",
    operation="OVERWRITE",
    rows_affected=target_count
)


log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'bronze_to_staging_market_finwire', 'Successfully completed processing for FINWIRE parsing.')
print(f"source_count : {source_count:,}")   
print(f"target_count : {target_count:,}")   